# CARCUX · B6 LLM baseline on Google Colab

Runs the LLM baseline (`ai/fusion/carcux_fusion/llm.py`) with **qwen2.5:7b** through Ollama on Colab's free GPU, on **simulated data only**. The model (4.7 GB) downloads onto Colab, not onto your PC.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. Allow Google Drive when asked: model answers are cached there, so a disconnect loses nothing; run all again and it carries on.
3. At the end, `b6-results.zip` downloads. Send its contents (or the printed tables) back.

Do not upload real CARCUX-BD reports here: real data stays on a local model.

## 1 · GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Save answers to Google Drive (survives disconnects)

In [ ]:
import os
CACHE = "/content/llm-cache"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = "/content/drive/MyDrive/carcux-b6/llm-cache"
except Exception as exc:  # Drive refused: answers stay in this session only
    print("Drive not mounted, cache is temporary:", exc)
os.makedirs(CACHE, exist_ok=True)
print("cache:", CACHE)

## 3 · Install Ollama, start it, download the model (a few minutes)

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama is running")
        break
    except Exception:
        time.sleep(1)
!ollama pull qwen2.5:7b
!ollama run qwen2.5:7b "Say hello in Bangla"


## 4 · Get CARCUX (the main branch) and install it

In [ ]:
%cd /content
!rm -rf CARCUX && git clone -q --depth 1 https://github.com/rakibuzzamanakash2-cyber/CARCUX.git
%cd /content/CARCUX/ai
!pip install -q -e ../data/tools -e correlation -e fusion
%cd /content/CARCUX/ai/fusion
!python -m carcux_fusion.llm --simulate --backend fake --cache "" --checkpoints final | tail -8

## 5 · Run B6

Each run: 60 seed events (about 85 with injected rumours), 4 checkpoints, 5-fold CV. The first run prints the speed every 25 calls; on a T4 expect a few seconds per call. `--no-stance` gives the model the text only.

In [ ]:
RUNS = [
    # (contamination, stance given)
    (0.3, True),
    (0.0, True),
    (0.6, True),
    (0.3, False),
]
os.makedirs("/content/b6", exist_ok=True)
for c, stance in RUNS:
    name = f"b6-c{int(c * 100)}-{'stance' if stance else 'text'}"
    flags = "" if stance else "--no-stance"
    print(f"\n===== {name} =====")
    !python -u -m carcux_fusion.llm --simulate --events 60 --contamination {c} {flags} --cache "{CACHE}" --json /content/b6/{name}.json | tee /content/b6/{name}.txt

## 6 · Download the results

In [ ]:
!cd /content && zip -qr b6-results.zip b6
from google.colab import files
files.download("/content/b6-results.zip")